# Lab 1: CIFAR10 Challenge

**CIFAR10** (http://www.cs.toronto.edu/~kriz/cifar.html) is one of the most famous ML data sets.

## Data
* 32x32 color images
* in 10 classes
* 50k training images
* 10k test images



<img src="https://docs.pytorch.org/tutorials/_images/cifar10.png" width=700>

In [2]:
#get data
from keras.datasets import cifar10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step


In [3]:
#traindata: 50k 32X32 rgb images
X_train.shape

(50000, 32, 32, 3)

In [4]:
#labels
y_train

array([[6],
       [9],
       [9],
       ...,
       [9],
       [1],
       [1]], dtype=uint8)

## Task: build the best classifier (with feature extration) using the methods you know from ML1+2
* work in small teams (2-4)
* use NumPy pre-processing, feature extraction and hyer-parameter tuning in Scikit-Learn
* no Neural Networks!
* best test F1-Score winns!
* remember: Feature engineering is the key ...

### Comparing different classifiers on plain mage vectors

In [7]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline

print("Starting classifier comparison...")

# --- 1. Data Preprocessing ---
# Flatten the images: convert 32x32x3 to 3072 features
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# Reshape y_train and y_test to be 1D arrays for Scikit-learn
y_train_flat = y_train.ravel()
y_test_flat = y_test.ravel()

print(f"X_train_flat shape: {X_train_flat.shape}")
print(f"X_test_flat shape: {X_test_flat.shape}")

# --- 2. Define Classifiers and Pipelines ---
# Using pipelines to include scaling
classifiers = {
    #"Logistic Regression": Pipeline([
    #    ('scaler', StandardScaler()),
    #    ('classifier', LogisticRegression(max_iter=100, random_state=42, solver='saga', n_jobs=-1))
    #]),
    "Random Forest": Pipeline([
        ('scaler', StandardScaler()), # Scaling might not be strictly necessary for tree-based models, but can't hurt
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ]),
    "Support Vector Machine": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', SVC(random_state=42, max_iter=100))
    ])
}

# --- 3. Train and Evaluate Classifiers ---
results = {}

for name, pipeline in classifiers.items():
    print(f"\nTraining {name}...")
    try:
        pipeline.fit(X_train_flat, y_train_flat)
        y_pred = pipeline.predict(X_test_flat)
        f1 = f1_score(y_test_flat, y_pred, average='weighted') # Use 'weighted' for imbalanced classes, though CIFAR-10 is balanced
        results[name] = f1
        print(f"  {name} F1-Score: {f1:.4f}")
        print(f"  {name} Classification Report:\n{classification_report(y_test_flat, y_pred)}")
    except Exception as e:
        print(f"Error training or evaluating {name}: {e}")
        results[name] = None

print("\n--- Comparison Results (F1-Score) ---")
for name, f1 in results.items():
    if f1 is not None:
        print(f"{name}: {f1:.4f}")
    else:
        print(f"{name}: Failed to compute")

print("Classifier comparison complete.")


Starting classifier comparison...
X_train_flat shape: (50000, 3072)
X_test_flat shape: (10000, 3072)

Training Random Forest...
  Random Forest F1-Score: 0.4630
  Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.55      0.56      0.55      1000
           1       0.52      0.55      0.53      1000
           2       0.37      0.34      0.35      1000
           3       0.33      0.28      0.30      1000
           4       0.39      0.39      0.39      1000
           5       0.43      0.40      0.41      1000
           6       0.47      0.56      0.51      1000
           7       0.50      0.45      0.47      1000
           8       0.58      0.61      0.59      1000
           9       0.47      0.53      0.50      1000

    accuracy                           0.47     10000
   macro avg       0.46      0.47      0.46     10000
weighted avg       0.46      0.47      0.46     10000


Training Support Vector Machine...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=100).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


  Support Vector Machine F1-Score: 0.2392
  Support Vector Machine Classification Report:
              precision    recall  f1-score   support

           0       0.19      0.46      0.27      1000
           1       0.33      0.29      0.31      1000
           2       0.14      0.20      0.16      1000
           3       0.22      0.15      0.18      1000
           4       0.33      0.15      0.20      1000
           5       0.21      0.09      0.13      1000
           6       0.34      0.36      0.35      1000
           7       0.22      0.20      0.21      1000
           8       0.34      0.43      0.38      1000
           9       0.34      0.16      0.21      1000

    accuracy                           0.25     10000
   macro avg       0.26      0.25      0.24     10000
weighted avg       0.26      0.25      0.24     10000


--- Comparison Results (F1-Score) ---
Random Forest: 0.4630
Support Vector Machine: 0.2392
Classifier comparison complete.


### SIFT Bag of Features Approach with RF

In [8]:
import cv2
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
import warnings

# Suppress UserWarnings from MiniBatchKMeans due to random state issues
warnings.filterwarnings("ignore", category=UserWarning, module='sklearn.cluster._kmeans')

print("Starting Bag of Features (BoF) approach with SIFT and Random Forest...")

# --- 1. SIFT Feature Extraction ---
# Initialize SIFT detector
sift = cv2.SIFT_create()

def extract_sift_descriptors(images):
    all_descriptors = []
    image_descriptors = []
    for i, img in enumerate(images):
        if i % 5000 == 0: # Print progress
            print(f"  Processing image {i}/{len(images)} for SIFT feature extraction...")
        # SIFT works on grayscale images
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        keypoints, descriptors = sift.detectAndCompute(gray_img, None)
        if descriptors is not None:
            all_descriptors.append(descriptors)
            image_descriptors.append(descriptors) # Store descriptors per image
        else:
            image_descriptors.append(np.array([]).reshape(0, 128)) # Handle images with no descriptors
    # Concatenate all descriptors for vocabulary building (only non-empty ones)
    if all_descriptors: # Ensure all_descriptors is not empty
        return np.vstack(all_descriptors), image_descriptors
    else:
        return np.array([]).reshape(0, 128), image_descriptors # Return empty array if no descriptors found

print("Extracting SIFT features from training images...")
# For computational efficiency, we might sample a subset of descriptors for KMeans
# For now, let's try with all and see if it's feasible.
all_train_descriptors, train_image_descriptors = extract_sift_descriptors(X_train)
print("Extracting SIFT features from test images...")
_, test_image_descriptors = extract_sift_descriptors(X_test)

print(f"Total SIFT descriptors from training images: {all_train_descriptors.shape}")

# --- 2. Build Visual Vocabulary (Codebook) using MiniBatchKMeans ---
num_visual_words = 200 # A common choice, can be tuned
print(f"Building visual vocabulary with {num_visual_words} visual words using MiniBatchKMeans...")
kmeans = MiniBatchKMeans(n_clusters=num_visual_words, random_state=42, n_init=10, batch_size=256)
kmeans.fit(all_train_descriptors)

# --- 3. Create Bag of Features (BoF) Histograms ---
def create_bof_histograms(image_descriptors, kmeans_model, num_words):
    bof_features = np.zeros((len(image_descriptors), num_words), dtype=np.float32)
    for i, descriptors in enumerate(image_descriptors):
        if descriptors.shape[0] > 0:
            # Predict the cluster for each descriptor in the image
            visual_words = kmeans_model.predict(descriptors)
            # Create a histogram of visual words
            for word_idx in visual_words:
                bof_features[i, word_idx] += 1
        # Normalize the histogram (L1 normalization)
        if np.sum(bof_features[i]) > 0:
            bof_features[i] = bof_features[i] / np.sum(bof_features[i])
    return bof_features

print("Creating BoF histograms for training data...")
X_train_bof = create_bof_histograms(train_image_descriptors, kmeans, num_visual_words)
print("Creating BoF histograms for test data...")
X_test_bof = create_bof_histograms(test_image_descriptors, kmeans, num_visual_words)

print(f"X_train_bof shape: {X_train_bof.shape}")
print(f"X_test_bof shape: {X_test_bof.shape}")

# Reshape y_train and y_test to be 1D arrays for Scikit-learn if not already
y_train_flat = y_train.ravel()
y_test_flat = y_test.ravel()

# --- 4. Train and Evaluate Random Forest Classifier on BoF Features ---
print("\nTraining Random Forest Classifier on BoF features...")
# Using a pipeline to include scaling
rf_bof_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

try:
    rf_bof_pipeline.fit(X_train_bof, y_train_flat)
    y_pred_bof = rf_bof_pipeline.predict(X_test_bof)
    f1_bof = f1_score(y_test_flat, y_pred_bof, average='weighted')
    print(f"  Random Forest (BoF SIFT) F1-Score: {f1_bof:.4f}")
    print(f"  Random Forest (BoF SIFT) Classification Report:\n{classification_report(y_test_flat, y_pred_bof)}")
except Exception as e:
    print(f"Error training or evaluating Random Forest (BoF SIFT): {e}")

print("BoF SIFT with Random Forest comparison complete.")


Starting Bag of Features (BoF) approach with SIFT and Random Forest...
Extracting SIFT features from training images...
  Processing image 0/50000 for SIFT feature extraction...
  Processing image 5000/50000 for SIFT feature extraction...
  Processing image 10000/50000 for SIFT feature extraction...
  Processing image 15000/50000 for SIFT feature extraction...
  Processing image 20000/50000 for SIFT feature extraction...
  Processing image 25000/50000 for SIFT feature extraction...
  Processing image 30000/50000 for SIFT feature extraction...
  Processing image 35000/50000 for SIFT feature extraction...
  Processing image 40000/50000 for SIFT feature extraction...
  Processing image 45000/50000 for SIFT feature extraction...
Extracting SIFT features from test images...
  Processing image 0/10000 for SIFT feature extraction...
  Processing image 5000/10000 for SIFT feature extraction...
Total SIFT descriptors from training images: (656497, 128)
Building visual vocabulary with 200 visual